In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [2]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [3]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [4]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def demand_driver_realign_pskus(data, channel):
    """
    Realign the old pskus to new pskus and return updated data.

    Args:
        data: pandas dataframe
        - master dataframe having all the pskus
    
    Return:
        data: pandas dataframe
        - dataframe 
    """
    realignment_data = realignment_df.copy()
    realignment_data.columns = realignment_data.columns.str.lower()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku

    return data


### Push heuristic data

In [ ]:
# df = pd.read_excel('/data/aman_singh/acuuracy_check/All_combination_Nov25_live_with_festivals((Autorecovered-312166393861403186)).xlsb', sheet_name = 'Base')
# df

,key,month_date,pred_prophet,pred_rf,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,...,RF_Recency NS Heuristic Val,RF_Final Heursitic Val,Remark,Missing,ALL Planning Principle,NON ALL Planning Principle,Skip basis PP,Heuristic > 2x Model,Diff,Comparison
0,BCE1_D231_718589,45991,5.264959,2.7,0.000237,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
1,BCE1_D231_718589,46022,4.547336,3.6,0.000205,0.000162,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
2,BCE1_D231_718589,46053,7.053803,2.7,0.000317,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
3,BCE1_D231_718589,46081,5.662848,1.8,0.000255,0.000081,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
4,BCE1_D231_718589,46112,0.000000,2.7,0.000000,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,1,0.0,P3M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCW2_D463_810125,46022,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164366,QCW2_D463_810125,46053,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164367,QCW2_D463_810125,46081,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164368,QCW2_D463_810125,46112,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M


In [ ]:
# df['month_date'] = (
#     pd.to_datetime(df['month_date'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )
# df['run_month'] = (
#     pd.to_datetime(df['run_month'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )

# df['month_date'] = (
#     pd.to_datetime(df['month_date']) + pd.offsets.MonthEnd(0)
# )
# df['run_month'] = (
#     pd.to_datetime(df['run_month']) + pd.offsets.MonthEnd(0)
# )

In [ ]:
# df['month_date'] = df['month_date'].astype(str)
# df['run_month'] = df['run_month'].astype(str)
# df.columns

Index(['key', 'month_date', 'pred_prophet', 'pred_rf', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month_value',
       'pred_best_model', 'pred_value_best_model',
       'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov',
       'run_month', 'M month', 'pred_prophet_70%ile', 'portfolio',
       'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M',
       'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2', 'Growth Flag',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'LY', 'LLY', 'LY value', 'LLY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3', 'ASM', 'Depot', 'PSKU', 'class', '

In [ ]:
# upload_df = df[['channel','portfolio', 'brand_code', 'class',
#         'run_month', 'M month','month_date', 'ASM', 'Depot', 'PSKU','pred_prophet', 'pred_rf','Final Heuristic 2 Vol',
#         'RF_Final Heuristic Vol']].rename(
#             columns = {'brand_code':'brand', 'class':'Brand Class', 'month_date':'month','pred_prophet':'prophet vol',
#                        'pred_rf':'rf_vol', 'Final Heuristic 2 Vol':'prophet heuristic vol',
#                        'RF_Final Heuristic Vol':'rf heuristic vol'}
#         )
# upload_df

,channel,portfolio,brand,Brand Class,run_month,M month,month,ASM,Depot,PSKU,prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.7,5.264959,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.6,4.547336,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.7,7.053803,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.8,5.662848,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.7,4.500000,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164366,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164367,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164368,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [ ]:
# upload_df.columns = upload_df.columns.str.upper()
# upload_df

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PROPHET VOL,RF_VOL,PROPHET HEURISTIC VOL,RF HEURISTIC VOL
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.7,5.264959,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.6,4.547336,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.7,7.053803,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.8,5.662848,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.7,4.500000,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164366,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164367,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164368,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [ ]:
# from snowflake.connector.pandas_tools import write_pandas

# write_pandas(dev_conn, upload_df, 
#             table_name = "TRN_MIL_DF_HEURISTICS_OUTPUT",
#             auto_create_table=True,
#             overwrite = False,)

(True,
 1,
 164370,
 [('rtthbeires/file0.txt',
   'LOADED',
   164370,
   164370,
   1,
   0,
   None,
   None,
   None,
   None)])

### Push shared output

In [22]:
df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/Stat Demand Forecast MT_as_on_9th_June_2026.xlsb',
    sheet_name='Base'
)


In [23]:
df['run_month'] = '2026-06-30'
df


,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,NPD Flag,run_month
0,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,46234,BCE1,D231,...,0.000347,0.000578,0.000459,0.000347,0.000289,0.000347,0.000555,False,NON-NPD,2026-06-30
1,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,46265,BCE1,D231,...,0.000694,0.000578,0.000459,0.000347,0.000289,0.000347,0.001111,False,NON-NPD,2026-06-30
2,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,46295,BCE1,D231,...,0.000000,0.000578,0.000459,0.000347,0.000405,0.000463,0.000000,False,NON-NPD,2026-06-30
3,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,46326,BCE1,D231,...,0.000347,0.000578,0.000459,0.000347,0.000347,0.000347,0.000555,False,NON-NPD,2026-06-30
4,MT,B2B,CNO,PCNO(R),A,349274.001420,BCE1_D231_718312,46234,BCE1,D231,...,0.002794,0.001397,0.001863,0.000000,0.000699,0.001397,0.002749,False,NON-NPD,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28107,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811219,46326,MCW2,D463,...,0.000000,0.000365,0.000000,0.000000,0.000000,0.000000,0.000365,False,NON-NPD,2026-06-30
28108,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,46234,MCW2,D463,...,0.000000,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30
28109,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,46265,MCW2,D463,...,0.000000,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30
28110,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,46295,MCW2,D463,...,0.000000,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30


In [24]:
df['Month'] = (
    pd.to_datetime(df['Month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,NPD Flag,run_month
0,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-07-01,BCE1,D231,...,0.000347,0.000578,0.000459,0.000347,0.000289,0.000347,0.000555,False,NON-NPD,2026-06-30
1,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-08-01,BCE1,D231,...,0.000694,0.000578,0.000459,0.000347,0.000289,0.000347,0.001111,False,NON-NPD,2026-06-30
2,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-09-01,BCE1,D231,...,0.000000,0.000578,0.000459,0.000347,0.000405,0.000463,0.000000,False,NON-NPD,2026-06-30
3,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-10-01,BCE1,D231,...,0.000347,0.000578,0.000459,0.000347,0.000347,0.000347,0.000555,False,NON-NPD,2026-06-30
4,MT,B2B,CNO,PCNO(R),A,349274.001420,BCE1_D231_718312,2026-07-01,BCE1,D231,...,0.002794,0.001397,0.001863,0.000000,0.000699,0.001397,0.002749,False,NON-NPD,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28107,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811219,2026-10-01,MCW2,D463,...,0.000000,0.000365,0.000000,0.000000,0.000000,0.000000,0.000365,False,NON-NPD,2026-06-30
28108,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-07-01,MCW2,D463,...,0.000000,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30
28109,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-08-01,MCW2,D463,...,0.000000,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30
28110,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-09-01,MCW2,D463,...,0.000000,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30


In [25]:
df.columns


Index(['Channel', 'Sub Channel', 'Portfolio', 'Brand', 'Brand Class',
       'Qtr Index Rate', 'Key', 'Month', 'ASM', 'Depot', 'PSKU',
       'LY Vol (ROUM)', 'P3M Vol (ROUM)', 'P6M Vol (ROUM)',
       'LY P3M Vol (ROUM)', 'LY P6M Vol (ROUM)', 'LY P3M (Rolling) Vol (ROUM)',
       'Pred Vol (ROUM)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
       'Sec Value Lag 3 (Cr)', 'LY Value Lag 1 (Cr)', 'LY Value Lag 2 (Cr)',
       'LY Value Lead 1 (Cr)', 'LY Value Lead 2 (Cr)', 'Planning Principle',
       'LY Val (Cr)', 'P3M Val (Cr)', 'P6M Val (Cr)', 'LY P3M Val (Cr)',
       'LY P6M Val (Cr)', 'LY P3M (Rolling) Val (Cr)', 'Pred Val (Cr)',
       'Primary P3M 0?', 'NPD Flag', 'run_month'],
      dtype='object')

In [ ]:
#df.drop(['Qtr Index Rate', 'Brand Class', 'LY (ROUM)', 'LY Value (in Cr)', 'Pred Value (in Cr)'], axis=1, inplace=True)

In [26]:
df['Month'] = (
    pd.to_datetime(df['Month']) + pd.offsets.MonthEnd(0)
)


In [27]:
df['run_month'] = pd.to_datetime(df['run_month'])
df['Month'] = pd.to_datetime(df['Month'])


mappings = {}

for run_month in df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-06-30 00:00:00'): {Timestamp('2026-06-30 00:00:00'): 'M',
  Timestamp('2026-07-31 00:00:00'): 'M+1',
  Timestamp('2026-08-31 00:00:00'): 'M+2',
  Timestamp('2026-09-30 00:00:00'): 'M+3',
  Timestamp('2026-10-31 00:00:00'): 'M+4',
  Timestamp('2026-11-30 00:00:00'): 'M+5',
  Timestamp('2026-12-31 00:00:00'): 'M+6',
  Timestamp('2027-01-31 00:00:00'): 'M+7',
  Timestamp('2027-02-28 00:00:00'): 'M+8'}}

In [28]:
df['M month'] = df.apply(
    lambda x: mappings[x['run_month']].get(
        x['Month']
    ), axis=1
)


In [29]:
df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,NPD Flag,run_month,M month
0,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-07-31,BCE1,D231,...,0.000578,0.000459,0.000347,0.000289,0.000347,0.000555,False,NON-NPD,2026-06-30,M+1
1,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-08-31,BCE1,D231,...,0.000578,0.000459,0.000347,0.000289,0.000347,0.001111,False,NON-NPD,2026-06-30,M+2
2,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-09-30,BCE1,D231,...,0.000578,0.000459,0.000347,0.000405,0.000463,0.000000,False,NON-NPD,2026-06-30,M+3
3,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-10-31,BCE1,D231,...,0.000578,0.000459,0.000347,0.000347,0.000347,0.000555,False,NON-NPD,2026-06-30,M+4
4,MT,B2B,CNO,PCNO(R),A,349274.001420,BCE1_D231_718312,2026-07-31,BCE1,D231,...,0.001397,0.001863,0.000000,0.000699,0.001397,0.002749,False,NON-NPD,2026-06-30,M+1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28107,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811219,2026-10-31,MCW2,D463,...,0.000365,0.000000,0.000000,0.000000,0.000000,0.000365,False,NON-NPD,2026-06-30,M+4
28108,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-07-31,MCW2,D463,...,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30,M+1
28109,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-08-31,MCW2,D463,...,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30,M+2
28110,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-09-30,MCW2,D463,...,0.000741,0.000000,0.000000,0.000000,0.000000,0.000741,False,NON-NPD,2026-06-30,M+3


In [30]:
df['Month'] = df['Month'].astype(str)
df['run_month'] = df['run_month'].astype(str)
df.columns

Index(['Channel', 'Sub Channel', 'Portfolio', 'Brand', 'Brand Class',
       'Qtr Index Rate', 'Key', 'Month', 'ASM', 'Depot', 'PSKU',
       'LY Vol (ROUM)', 'P3M Vol (ROUM)', 'P6M Vol (ROUM)',
       'LY P3M Vol (ROUM)', 'LY P6M Vol (ROUM)', 'LY P3M (Rolling) Vol (ROUM)',
       'Pred Vol (ROUM)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
       'Sec Value Lag 3 (Cr)', 'LY Value Lag 1 (Cr)', 'LY Value Lag 2 (Cr)',
       'LY Value Lead 1 (Cr)', 'LY Value Lead 2 (Cr)', 'Planning Principle',
       'LY Val (Cr)', 'P3M Val (Cr)', 'P6M Val (Cr)', 'LY P3M Val (Cr)',
       'LY P6M Val (Cr)', 'LY P3M (Rolling) Val (Cr)', 'Pred Val (Cr)',
       'Primary P3M 0?', 'NPD Flag', 'run_month', 'M month'],
      dtype='object')

In [31]:
upload_df = df[['Channel','Portfolio', 'Brand', 'Brand Class',
        'run_month', 'M month','Month', 'ASM', 'Depot', 'PSKU','Pred Vol (ROUM)']]


In [32]:
upload_df

,Channel,Portfolio,Brand,Brand Class,run_month,M month,Month,ASM,Depot,PSKU,Pred Vol (ROUM)
0,MT,Others,REV.LQDST,B,2026-06-30,M+1,2026-07-31,BCE1,D231,718303,0.022400
1,MT,Others,REV.LQDST,B,2026-06-30,M+2,2026-08-31,BCE1,D231,718303,0.044800
2,MT,Others,REV.LQDST,B,2026-06-30,M+3,2026-09-30,BCE1,D231,718303,0.000000
3,MT,Others,REV.LQDST,B,2026-06-30,M+4,2026-10-31,BCE1,D231,718303,0.022400
4,MT,CNO,PCNO(R),A,2026-06-30,M+1,2026-07-31,BCE1,D231,718312,0.078703
...,...,...,...,...,...,...,...,...,...,...,...
28107,MT,Foods,TRU_ELMNT,NPD,2026-06-30,M+4,2026-10-31,MCW2,D463,811219,0.010333
28108,MT,Foods,TRU_ELMNT,NPD,2026-06-30,M+1,2026-07-31,MCW2,D463,811220,0.021000
28109,MT,Foods,TRU_ELMNT,NPD,2026-06-30,M+2,2026-08-31,MCW2,D463,811220,0.021000
28110,MT,Foods,TRU_ELMNT,NPD,2026-06-30,M+3,2026-09-30,MCW2,D463,811220,0.021000


In [ ]:
# upload_df[upload_df['Month'] == '2026-04-30']['Pred Vol (ROUM)'].sum()

5346018.0036901245

In [33]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PRED VOL (ROUM)
0,MT,Others,REV.LQDST,B,2026-06-30,M+1,2026-07-31,BCE1,D231,718303,0.022400
1,MT,Others,REV.LQDST,B,2026-06-30,M+2,2026-08-31,BCE1,D231,718303,0.044800
2,MT,Others,REV.LQDST,B,2026-06-30,M+3,2026-09-30,BCE1,D231,718303,0.000000
3,MT,Others,REV.LQDST,B,2026-06-30,M+4,2026-10-31,BCE1,D231,718303,0.022400
4,MT,CNO,PCNO(R),A,2026-06-30,M+1,2026-07-31,BCE1,D231,718312,0.078703
...,...,...,...,...,...,...,...,...,...,...,...
28107,MT,Foods,TRU_ELMNT,NPD,2026-06-30,M+4,2026-10-31,MCW2,D463,811219,0.010333
28108,MT,Foods,TRU_ELMNT,NPD,2026-06-30,M+1,2026-07-31,MCW2,D463,811220,0.021000
28109,MT,Foods,TRU_ELMNT,NPD,2026-06-30,M+2,2026-08-31,MCW2,D463,811220,0.021000
28110,MT,Foods,TRU_ELMNT,NPD,2026-06-30,M+3,2026-09-30,MCW2,D463,811220,0.021000


In [34]:
upload_df.dtypes

CHANNEL             object
PORTFOLIO           object
BRAND               object
BRAND CLASS         object
RUN_MONTH           object
M MONTH             object
MONTH               object
ASM                 object
DEPOT               object
PSKU                 int64
PRED VOL (ROUM)    float64
dtype: object

In [35]:
upload_df['PORTFOLIO'] = upload_df['PORTFOLIO'].astype(str)


In [36]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_SHARED",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 28112,
 [('qpssapvfpc/file0.txt',
   'LOADED',
   28112,
   28112,
   1,
   0,
   None,
   None,
   None,
   None)])

### push offtakes to primary

In [37]:
qcom_df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/Ecom_depot_psku_forecast_as_on_10th_june_2026.xlsx',
    sheet_name='Base'
)

In [38]:
qcom_df

,Key,Depot,PSKU,PSKU Description,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,...,LY Primary Actuals Lag 3 Val,LY Primary Actuals Lead 1 Val,LY Primary Actuals Lead 2 Val,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val,Brand Class,NPD Flag,Planning Principle,Primary P3M 0?
0,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-06-30,2026-07-31,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,A,NON-NPD,Valid,True
1,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-06-30,2026-08-31,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,A,NON-NPD,Valid,True
2,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-06-30,2026-09-30,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,A,NON-NPD,Valid,True
3,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-06-30,2026-10-31,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,A,NON-NPD,Valid,True
4,D111_710981,D111,710981,PA Men 200ml Almond Hair Oil BT,PA_MEN_AH,2026-06-30,2026-07-31,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77839,D677_811287,D677,811287,PA ROSEMARY HAIR SPRAY 100ML,PA_RSW_SR,2026-06-30,2026-10-31,NaN,0,0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NPD,Valid,True
77840,D677_811416,D677,811416,SAF MILLET MUESLI 650GM POUCH,SAF-MUSLI,2026-06-30,2026-07-31,0.0,0,0,...,NaN,NaN,NaN,NaN,0.0,0.0,NPD,NON-NPD,Valid,True
77841,D677_811416,D677,811416,SAF MILLET MUESLI 650GM POUCH,SAF-MUSLI,2026-06-30,2026-08-31,0.0,0,0,...,NaN,NaN,NaN,NaN,0.0,0.0,NPD,NON-NPD,Valid,True
77842,D677_811416,D677,811416,SAF MILLET MUESLI 650GM POUCH,SAF-MUSLI,2026-06-30,2026-09-30,0.0,0,0,...,NaN,NaN,NaN,NaN,0.0,0.0,NPD,NON-NPD,Valid,True


In [ ]:
# qcom_df['Month Date'] = (
#     pd.to_datetime(qcom_df['Month Date']) + pd.offsets.MonthEnd(0)
# )

In [ ]:
# qcom_df['Run Month'] = (
#     pd.to_datetime(qcom_df['Run Month'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )
# qcom_df['Run Month'] = (
#     pd.to_datetime(qcom_df['Run Month']) + pd.offsets.MonthEnd(0)
# )
# qcom_df

,Key,Depot,PSKU,PSKU Desc,Brand,Portfolio,Index Rate,Run Month,Month Date,M Month,...,LY Offtake Chain FC PSKU Lead 2 Val,Actual Closing SOH Val,Actual Closing SOH Lag 1 Val,Actual Closing SOH Lag 2 Val,Final Assumed Closing SOH Val,Final Assumed Closing SOH Lag 1 Val,Safety Stock Val,Brand Class,Planning Principle,Primary P3M 0?
0,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-03-31,M+1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
1,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-04-30,M+2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
2,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-05-31,M+3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
3,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-06-30,M+4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
4,D112_718287,D112,718287,PCNO 200ml JAR,PCNO(R),CNO,309765.865129,2026-02-28,2026-03-31,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A,Valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30603,D677_810805,D677,810805,PA BABY FACE BODY WIPES 1KG,PABABY_GM,Skin Care,451.133000,2026-02-28,2026-06-30,M+4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30604,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-03-31,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30605,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-04-30,M+2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30606,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-05-31,M+3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True


In [39]:
qcom_df['Run Month'] = pd.to_datetime(qcom_df['Run Month'])
qcom_df['Month Date'] = pd.to_datetime(qcom_df['Month Date'])

mappings ={}

for run_month in qcom_df['Run Month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


   
qcom_df['M month'] = qcom_df.apply(
    lambda x: mappings[x['Run Month']].get(
        x['Month Date']
    ), axis=1
)


In [40]:
qcom_df.columns

Index(['Key', 'Depot', 'PSKU', 'PSKU Description', 'Brand', 'Run Month',
       'Month Date', 'Calculated PSKU Primary Vol', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol', 'Primary P3M copy Vol',
       'PSKU Primary P3M Sum Vol', 'PSKU P3M Contribution',
       'Calculated Depot PSKU Primary Vol', 'Index Rate',
       'Calculated PSKU Primary Val', 'Primary Actuals Val',
       'Primary Plan Val', 'Secondary Plan Val', 'Secondary Actuals Val',
       'Primary P3M Val', 'Primary Actuals Lag 1 Val',
       'Primary Actuals Lag 2 Val', 'Primary Actuals Lag 3 Val',
       'LY Primary Actu

In [41]:
qcom_df.rename(columns = {'Calculated Depot PSKU Primary Vol':'Calculated Primary Vol'}, inplace = True)


In [42]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
brand_md_df.rename(columns = {'brand_code':'Brand'}, inplace = True)

In [43]:

len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    brand_md_df,
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(qcom_df)

In [44]:
qcom_df

,Key,Depot,PSKU,PSKU Description,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,...,LY Primary Actuals Lead 2 Val,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val,Brand Class,NPD Flag,Planning Principle,Primary P3M 0?,M month,portfolio
0,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-06-30,2026-07-31,NaN,0,0,...,0.0,0.0,0.0,0.0,A,NON-NPD,Valid,True,M+1,Foods
1,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-06-30,2026-08-31,NaN,0,0,...,0.0,0.0,0.0,0.0,A,NON-NPD,Valid,True,M+2,Foods
2,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-06-30,2026-09-30,NaN,0,0,...,0.0,0.0,0.0,0.0,A,NON-NPD,Valid,True,M+3,Foods
3,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-06-30,2026-10-31,NaN,0,0,...,0.0,0.0,0.0,0.0,A,NON-NPD,Valid,True,M+4,Foods
4,D111_710981,D111,710981,PA Men 200ml Almond Hair Oil BT,PA_MEN_AH,2026-06-30,2026-07-31,NaN,0,0,...,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,True,M+1,Hair Oils
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77839,D677_811287,D677,811287,PA ROSEMARY HAIR SPRAY 100ML,PA_RSW_SR,2026-06-30,2026-10-31,NaN,0,0,...,NaN,NaN,0.0,0.0,NaN,NPD,Valid,True,M+4,NaN
77840,D677_811416,D677,811416,SAF MILLET MUESLI 650GM POUCH,SAF-MUSLI,2026-06-30,2026-07-31,0.0,0,0,...,NaN,NaN,0.0,0.0,NPD,NON-NPD,Valid,True,M+1,Foods
77841,D677_811416,D677,811416,SAF MILLET MUESLI 650GM POUCH,SAF-MUSLI,2026-06-30,2026-08-31,0.0,0,0,...,NaN,NaN,0.0,0.0,NPD,NON-NPD,Valid,True,M+2,Foods
77842,D677_811416,D677,811416,SAF MILLET MUESLI 650GM POUCH,SAF-MUSLI,2026-06-30,2026-09-30,0.0,0,0,...,NaN,NaN,0.0,0.0,NPD,NON-NPD,Valid,True,M+3,Foods


In [45]:
qcom_df = qcom_df.groupby(['Month Date', 'Depot', 'PSKU',
       'Brand', 'portfolio','M month','Run Month'])[['Calculated Primary Vol']].sum().reset_index()
qcom_df

,Month Date,Depot,PSKU,Brand,portfolio,M month,Run Month,Calculated Primary Vol
0,2026-07-31,D111,709567,SAFF OATS,Foods,M+1,2026-06-30,0.0
1,2026-07-31,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-06-30,0.0
2,2026-07-31,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
3,2026-07-31,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
4,2026-07-31,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
...,...,...,...,...,...,...,...,...
74523,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74524,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74525,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74526,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0


In [46]:

qcom_df.rename(columns = {'Month Date':'Month', 'Final PSKU':'PSKU', 'Depot Code':'Depot', 'Run Month':'run_month','portfolio':'Portfolio'}, inplace = True)
qcom_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-07-31,D111,709567,SAFF OATS,Foods,M+1,2026-06-30,0.0
1,2026-07-31,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-06-30,0.0
2,2026-07-31,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
3,2026-07-31,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
4,2026-07-31,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
...,...,...,...,...,...,...,...,...
74523,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74524,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74525,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74526,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0


In [48]:
qcom_df[qcom_df['Month'] == '2026-07-31']['Calculated Primary Vol'].sum()

312575.5676821776

In [49]:
qcom_df['Month'] = qcom_df['Month'].astype(str)
qcom_df['run_month'] = qcom_df['run_month'].astype(str)
qcom_df.columns

Index(['Month', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol'],
      dtype='object')

In [50]:
qcom_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-07-31,D111,709567,SAFF OATS,Foods,M+1,2026-06-30,0.0
1,2026-07-31,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-06-30,0.0
2,2026-07-31,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
3,2026-07-31,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
4,2026-07-31,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0
...,...,...,...,...,...,...,...,...
74523,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74524,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74525,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0
74526,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0


In [51]:
upload_df = qcom_df[['Month', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol']]


In [52]:
upload_df['Channel'] = 'ECOM'

In [53]:
upload_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol,Channel
0,2026-07-31,D111,709567,SAFF OATS,Foods,M+1,2026-06-30,0.0,ECOM
1,2026-07-31,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-06-30,0.0,ECOM
2,2026-07-31,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0,ECOM
3,2026-07-31,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0,ECOM
4,2026-07-31,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0,ECOM
...,...,...,...,...,...,...,...,...,...
74523,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
74524,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
74525,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
74526,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0,ECOM


In [54]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH,DEPOT,PSKU,BRAND,PORTFOLIO,M MONTH,RUN_MONTH,CALCULATED PRIMARY VOL,CHANNEL
0,2026-07-31,D111,709567,SAFF OATS,Foods,M+1,2026-06-30,0.0,ECOM
1,2026-07-31,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-06-30,0.0,ECOM
2,2026-07-31,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0,ECOM
3,2026-07-31,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0,ECOM
4,2026-07-31,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-06-30,0.0,ECOM
...,...,...,...,...,...,...,...,...,...
74523,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
74524,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
74525,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
74526,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0,ECOM


In [55]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFT2PRIM_SHARED",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 74528,
 [('feumzyefsx/file0.txt',
   'LOADED',
   74528,
   74528,
   1,
   0,
   None,
   None,
   None,
   None)])

### Push chain psku primary

In [74]:
qcom_df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/Qcom_chain_depot_psku_2.xlsx',
    sheet_name='Base'
)

In [75]:
qcom_df

,Key,Chain,Depot,PSKU,PSKU Description,Brand,Index Rate,Portfolio,Run Month,Month Date,...,NPD Tag,Planning Principle,Primary P3M 0?,key,offtakes_Mar_vol,offtakes_Apr_vol,offtakes_May_vol,offtakes_Mar_Val,offtakes_Apr_Val,offtakes_May_Val
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-07-31,...,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-08-31,...,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-09-30,...,NON-NPD,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN
3,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-07-31,...,NON-NPD,Valid,True,blinkit_d112_718288,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-08-31,...,NON-NPD,Valid,True,blinkit_d112_718288,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63388,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-08-31,...,NPD,Valid,True,zepto_d674_811269,NaN,NaN,NaN,NaN,NaN,NaN
63389,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-09-30,...,NPD,Valid,True,zepto_d674_811269,NaN,NaN,NaN,NaN,NaN,NaN
63390,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-07-31,...,NON-NPD,Valid,True,zepto_d674_811279,NaN,NaN,NaN,NaN,NaN,NaN
63391,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-08-31,...,NON-NPD,Valid,True,zepto_d674_811279,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# qcom_df['Month Date'] = (
#     pd.to_datetime(qcom_df['Month Date']) + pd.offsets.MonthEnd(0)
# )

In [ ]:
# qcom_df['Run Month'] = (
#     pd.to_datetime(qcom_df['Run Month'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )
# qcom_df['Run Month'] = (
#     pd.to_datetime(qcom_df['Run Month']) + pd.offsets.MonthEnd(0)
# )
# qcom_df

,Key,Chain,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,...,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val,Brand Class,Planning Principle,Primary P3M 0?
0,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-03-31,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
1,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-04-30,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
2,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-05-31,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
3,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-06-30,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
4,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,2026-02-28,2026-03-31,M+1,...,0.28265,0.165361,0.190966,0.22004,0.159827,0.200795,0.34632,A,Valid,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11487,Nykaa_811169,Nykaa,811169,SW MINI EDP GIFT PO4 18ML,SW_SGPRF,1443.400363,Male Grooming,2026-02-28,2026-06-30,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11488,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-03-31,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11489,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-04-30,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11490,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-05-31,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True


In [76]:
qcom_df['Run Month'] = pd.to_datetime(qcom_df['Run Month'])
qcom_df['Month Date'] = pd.to_datetime(qcom_df['Month Date'])

mappings ={}

for run_month in qcom_df['Run Month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


   
qcom_df['M month'] = qcom_df.apply(
    lambda x: mappings[x['Run Month']].get(
        x['Month Date']
    ), axis=1
)


In [ ]:
# qcom_df = qcom_df[qcom_df['M month'] == 'M+1']
# qcom_df

,key2,key,Chain,FC,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,...,Offtake Chain FC PSKU Lag 3 Vol,LY Offtake Actuals Chain FC PSKU Vol,LY Offtake Chain FC PSKU P3M Vol,Offtake Chain FC PSKU P3M Vol,Offtake Chain FC PSKU Lag 1 Val,Offtake Chain FC PSKU Lag 2 Val,Offtake Chain FC PSKU Lag 3 Val,LY Offtake Actuals Chain FC PSKU Val,LY Offtake Chain FC PSKU P3M Val,Offtake Chain FC PSKU P3M Val
2,Blinkit_ahmedabad a2 - feeder warehouse_718288,Blinkit_718288,Blinkit,ahmedabad a2 - feeder warehouse,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,2025-11-30,...,6.873931,5.219777,3.804556,6.602009,0.107562,0.085510,0.094629,0.071857,0.052375,0.090885
9,Blinkit_ahmedabad a2 - feeder warehouse_718299,Blinkit_718299,Blinkit,ahmedabad a2 - feeder warehouse,718299,PCNO 100ml BTL,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
16,Blinkit_ahmedabad a2 - feeder warehouse_718308,Blinkit_718308,Blinkit,ahmedabad a2 - feeder warehouse,718308,PCNO 100ml JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
23,Blinkit_ahmedabad a2 - feeder warehouse_718310,Blinkit_718310,Blinkit,ahmedabad a2 - feeder warehouse,718310,PCNO 500ml JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
30,Blinkit_ahmedabad a2 - feeder warehouse_718312,Blinkit_718312,Blinkit,ahmedabad a2 - feeder warehouse,718312,PCNO 1L JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.196489,0.114393,0.065902,0.201895,0.006292,0.005917,0.006087,0.003543,0.002041,0.006254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235825,Zepto_pun-dry-mh2-koregaon_810673,Zepto_810673,Zepto,pun-dry-mh2-koregaon,810673,PA ESS ROSEMARY OIL 14ML,PA_ESS_HO,12900.000000,Hair Oils,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235832,Zepto_pun-dry-mh2-koregaon_810674,Zepto_810674,Zepto,pun-dry-mh2-koregaon,810674,PA ESS TEA TREE OIL 14ML,PA_ESS_HO,12900.000000,Hair Oils,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235839,Zepto_pun-dry-mh2-koregaon_810685,Zepto_810685,Zepto,pun-dry-mh2-koregaon,810685,SF MUESLI MANGO 400G POUCH,SAF-MUSLI,321959.667548,Foods,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235846,Zepto_pun-dry-mh2-koregaon_810738,Zepto_810738,Zepto,pun-dry-mh2-koregaon,810738,PA BABY FACE BODY WIPE 362GM,PABABY_GM,451.133000,Skin Care,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [77]:
qcom_df.columns

Index(['Key', 'Chain', 'Depot', 'PSKU', 'PSKU Description', 'Brand',
       'Index Rate', 'Portfolio', 'Run Month', 'Month Date',
       ...
       'NPD Tag', 'Planning Principle', 'Primary P3M 0?', 'key',
       'offtakes_Mar_vol', 'offtakes_Apr_vol', 'offtakes_May_vol',
       'offtakes_Mar_Val', 'offtakes_Apr_Val', 'offtakes_May_Val'],
      dtype='object', length=135)

In [78]:
qcom_df.rename(columns = {'Calculated Depot PSKU Primary Vol':'Calculated Primary Vol'}, inplace = True)


In [79]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
brand_md_df.rename(columns = {'brand_code':'Brand'}, inplace = True)

In [80]:

len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    brand_md_df,
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(qcom_df)

In [81]:
qcom_df

,Key,Chain,Depot,PSKU,PSKU Description,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Planning Principle,Primary P3M 0?,key,offtakes_Mar_vol,offtakes_Apr_vol,offtakes_May_vol,offtakes_Mar_Val,offtakes_Apr_Val,offtakes_May_Val,portfolio
0,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-07-31,...,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN,CNO
1,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-08-31,...,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN,CNO
2,blinkit_d112_718287,Blinkit,d112,718287,PCNO 200ml JAR,PCNO(R),349274.001420,CNO,2026-06-30,2026-09-30,...,Valid,True,blinkit_d112_718287,NaN,NaN,NaN,NaN,NaN,NaN,CNO
3,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-07-31,...,Valid,True,blinkit_d112_718288,0.0,0.0,0.0,0.0,0.0,0.0,Saffola Oils
4,blinkit_d112_718288,Blinkit,d112,718288,SAFF GOLD 5L JAR,SAFF GOLD,138865.260689,Saffola Oils,2026-06-30,2026-08-31,...,Valid,True,blinkit_d112_718288,0.0,0.0,0.0,0.0,0.0,0.0,Saffola Oils
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63388,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-08-31,...,Valid,True,zepto_d674_811269,NaN,NaN,NaN,NaN,NaN,NaN,Hair Oils
63389,zepto_d674_811269,Zepto,d674,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-06-30,2026-09-30,...,Valid,True,zepto_d674_811269,NaN,NaN,NaN,NaN,NaN,NaN,Hair Oils
63390,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-07-31,...,Valid,True,zepto_d674_811279,NaN,NaN,NaN,NaN,NaN,NaN,Saffola Oils
63391,zepto_d674_811279,Zepto,d674,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-06-30,2026-08-31,...,Valid,True,zepto_d674_811279,NaN,NaN,NaN,NaN,NaN,NaN,Saffola Oils


In [82]:
qcom_df = qcom_df.groupby(['Month Date', 'Chain', 'PSKU',
       'Brand', 'portfolio','M month','Run Month'])[['Calculated Primary Vol']].sum().reset_index()
qcom_df

,Month Date,Chain,PSKU,Brand,portfolio,M month,Run Month,Calculated Primary Vol
0,2026-07-31,Blinkit,718287,PCNO(R),CNO,M+1,2026-06-30,0.000000
1,2026-07-31,Blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-06-30,68.144673
2,2026-07-31,Blinkit,718297,PCNO(R),CNO,M+1,2026-06-30,0.000000
3,2026-07-31,Blinkit,718299,PCNO(R),CNO,M+1,2026-06-30,0.000000
4,2026-07-31,Blinkit,718300,PCNO FLEX,CNO,M+1,2026-06-30,0.000000
...,...,...,...,...,...,...,...,...
4954,2026-09-30,Zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000
4955,2026-09-30,Zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668
4956,2026-09-30,Zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000
4957,2026-09-30,Zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000


In [83]:

qcom_df.rename(columns = {'Month Date':'Month', 'Final PSKU':'PSKU', 'Depot Code':'Depot', 'Run Month':'run_month','portfolio':'Portfolio'}, inplace = True)
qcom_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-07-31,Blinkit,718287,PCNO(R),CNO,M+1,2026-06-30,0.000000
1,2026-07-31,Blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-06-30,68.144673
2,2026-07-31,Blinkit,718297,PCNO(R),CNO,M+1,2026-06-30,0.000000
3,2026-07-31,Blinkit,718299,PCNO(R),CNO,M+1,2026-06-30,0.000000
4,2026-07-31,Blinkit,718300,PCNO FLEX,CNO,M+1,2026-06-30,0.000000
...,...,...,...,...,...,...,...,...
4954,2026-09-30,Zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000
4955,2026-09-30,Zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668
4956,2026-09-30,Zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000
4957,2026-09-30,Zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000


In [84]:
qcom_df[qcom_df['Month'] == '2026-07-31']['Calculated Primary Vol'].sum()

141496.4618503959

In [85]:
qcom_df['Month'] = qcom_df['Month'].astype(str)
qcom_df['run_month'] = qcom_df['run_month'].astype(str)
qcom_df.columns

Index(['Month', 'Chain', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol'],
      dtype='object')

In [86]:
qcom_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-07-31,Blinkit,718287,PCNO(R),CNO,M+1,2026-06-30,0.000000
1,2026-07-31,Blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-06-30,68.144673
2,2026-07-31,Blinkit,718297,PCNO(R),CNO,M+1,2026-06-30,0.000000
3,2026-07-31,Blinkit,718299,PCNO(R),CNO,M+1,2026-06-30,0.000000
4,2026-07-31,Blinkit,718300,PCNO FLEX,CNO,M+1,2026-06-30,0.000000
...,...,...,...,...,...,...,...,...
4954,2026-09-30,Zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000
4955,2026-09-30,Zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668
4956,2026-09-30,Zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000
4957,2026-09-30,Zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000


In [87]:
upload_df = qcom_df[['Month', 'Chain', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol']]


In [88]:
upload_df['Channel'] = 'Qcom'

In [89]:
upload_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol,Channel
0,2026-07-31,Blinkit,718287,PCNO(R),CNO,M+1,2026-06-30,0.000000,Qcom
1,2026-07-31,Blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-06-30,68.144673,Qcom
2,2026-07-31,Blinkit,718297,PCNO(R),CNO,M+1,2026-06-30,0.000000,Qcom
3,2026-07-31,Blinkit,718299,PCNO(R),CNO,M+1,2026-06-30,0.000000,Qcom
4,2026-07-31,Blinkit,718300,PCNO FLEX,CNO,M+1,2026-06-30,0.000000,Qcom
...,...,...,...,...,...,...,...,...,...
4954,2026-09-30,Zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,Qcom
4955,2026-09-30,Zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,Qcom
4956,2026-09-30,Zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,Qcom
4957,2026-09-30,Zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,Qcom


In [90]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH,CHAIN,PSKU,BRAND,PORTFOLIO,M MONTH,RUN_MONTH,CALCULATED PRIMARY VOL,CHANNEL
0,2026-07-31,Blinkit,718287,PCNO(R),CNO,M+1,2026-06-30,0.000000,Qcom
1,2026-07-31,Blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2026-06-30,68.144673,Qcom
2,2026-07-31,Blinkit,718297,PCNO(R),CNO,M+1,2026-06-30,0.000000,Qcom
3,2026-07-31,Blinkit,718299,PCNO(R),CNO,M+1,2026-06-30,0.000000,Qcom
4,2026-07-31,Blinkit,718300,PCNO FLEX,CNO,M+1,2026-06-30,0.000000,Qcom
...,...,...,...,...,...,...,...,...,...
4954,2026-09-30,Zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,Qcom
4955,2026-09-30,Zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,Qcom
4956,2026-09-30,Zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,Qcom
4957,2026-09-30,Zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,Qcom


In [91]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFT2PRIM_CPSKU",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 4959,
 [('rtftrtakah/file0.txt',
   'LOADED',
   4959,
   4959,
   1,
   0,
   None,
   None,
   None,
   None)])

### push offtakes data

In [105]:
offtakes_df = pd.read_excel('/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_june_live.xlsx', sheet_name = 'base')

In [106]:
offtakes_df.columns[50:]

Index(['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3',
       'skipped', 'event_month_flag', 'event_uplift_factor',
       'event_sensitive_flag', 'P3M_adj', 'P6M_adj', 'P3M_adj_value',
       ...
       'final Heuristic vol', 'pred_SARIMA', 'pred_value_SARIMA',
       'shrink_ratio_sarima', 'recency_heuristic_sarima',
       'seasonal_heuristic_sarima', 'non_seasonal_heuristic_sarima',
       'final_heuristic_sarima', 'final_heuristic_sarima_value',
       'final_heuristic_sarima_value_2'],
      dtype='object', length=110)

In [107]:
offtakes_df.rename(columns = {'run_month_x':'run_month'}, inplace = True)
offtakes_df['run_month'].unique()

<DatetimeArray>
['2026-06-30 00:00:00']
Length: 1, dtype: datetime64[ns]

In [109]:
offtakes_df = offtakes_df.groupby(['month_date','platform_name', 'parent_material_code', 'brand_code','portfolio','run_month', 'M month']
                    )[['pred_prophet', 'pred_rf','Final Heuristic value']].sum().reset_index()

In [110]:
offtakes_df.rename(columns = {'Final Heuristic value':'final_heuristic_prophet_value_2'}, inplace = True)

In [111]:
offtakes_df['month_date'] = offtakes_df['month_date'].astype(str)
offtakes_df['run_month'] = offtakes_df['run_month'].astype(str)
offtakes_df.columns

Index(['month_date', 'platform_name', 'parent_material_code', 'brand_code',
       'portfolio', 'run_month', 'M month', 'pred_prophet', 'pred_rf',
       'final_heuristic_prophet_value_2'],
      dtype='object')

In [112]:
upload_df = offtakes_df.copy()

In [113]:
upload_df

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,M month,pred_prophet,pred_rf,final_heuristic_prophet_value_2
0,2026-06-30,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,2026-06-30,M,28.546677,25.025422,0.392938
1,2026-06-30,Amazon ARIPL,718321,SAFF KO,Saffola Oils,2026-06-30,M,0.000000,0.000000,0.000000
2,2026-06-30,Amazon ARIPL,718322,SAFF KO,Saffola Oils,2026-06-30,M,16.189676,16.358490,0.280468
3,2026-06-30,Amazon ARIPL,718323,SF_IMV_MK,Foods,2026-06-30,M,0.000000,0.000000,0.000000
4,2026-06-30,Amazon ARIPL,718328,SAFF KOCO,Saffola Oils,2026-06-30,M,11.115950,9.330333,0.129303
...,...,...,...,...,...,...,...,...,...,...
27015,2027-03-31,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-06-30,M+9,0.000000,0.000000,0.002317
27016,2027-03-31,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-06-30,M+9,0.000000,0.000000,0.000672
27017,2027-03-31,Nykaa,810738,PABABY_GM,Skin Care,2026-06-30,M+9,0.000000,0.000000,0.000000
27018,2027-03-31,Nykaa,810805,PABABY_GM,Skin Care,2026-06-30,M+9,0.000000,0.000000,0.000544


In [114]:
upload_df['channel'] = 'ECOM'

In [115]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,PORTFOLIO,RUN_MONTH,M MONTH,PRED_PROPHET,PRED_RF,FINAL_HEURISTIC_PROPHET_VALUE_2,CHANNEL
0,2026-06-30,Amazon ARIPL,718288,SAFF GOLD,Saffola Oils,2026-06-30,M,28.546677,25.025422,0.392938,ECOM
1,2026-06-30,Amazon ARIPL,718321,SAFF KO,Saffola Oils,2026-06-30,M,0.000000,0.000000,0.000000,ECOM
2,2026-06-30,Amazon ARIPL,718322,SAFF KO,Saffola Oils,2026-06-30,M,16.189676,16.358490,0.280468,ECOM
3,2026-06-30,Amazon ARIPL,718323,SF_IMV_MK,Foods,2026-06-30,M,0.000000,0.000000,0.000000,ECOM
4,2026-06-30,Amazon ARIPL,718328,SAFF KOCO,Saffola Oils,2026-06-30,M,11.115950,9.330333,0.129303,ECOM
...,...,...,...,...,...,...,...,...,...,...,...
27015,2027-03-31,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-06-30,M+9,0.000000,0.000000,0.002317,ECOM
27016,2027-03-31,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-06-30,M+9,0.000000,0.000000,0.000672,ECOM
27017,2027-03-31,Nykaa,810738,PABABY_GM,Skin Care,2026-06-30,M+9,0.000000,0.000000,0.000000,ECOM
27018,2027-03-31,Nykaa,810805,PABABY_GM,Skin Care,2026-06-30,M+9,0.000000,0.000000,0.000544,ECOM


In [116]:
upload_df[upload_df['MONTH_DATE'] == '2026-07-31']['FINAL_HEURISTIC_PROPHET_VALUE_2'].sum()

44.687661582339715

In [117]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFFTAKES_OUTPUT",
            auto_create_table=True,
            overwrite = False)

(True,
 1,
 27020,
 [('qmrlgkkoel/file0.txt',
   'LOADED',
   27020,
   27020,
   1,
   0,
   None,
   None,
   None,
   None)])